# Apache Airflow Core Concepts & Enterprise Pipeline Patterns

Welcome to **Project 08: Apache Airflow for Data Engineering** in the *Data Engineering Diaries* series by Sathiya Murthy.

This interactive notebook covers essential Apache Airflow architectural concepts, code patterns, and practical execution techniques:
1. **Airflow Architecture**: Webserver, Scheduler, Metadata Database, Executor, Workers, Triggerer
2. **TaskFlow API vs Traditional Operators**: Modern `@dag` and `@task` paradigms vs `PythonOperator`/`BashOperator`
3. **Dynamic Task Mapping (`.expand()`)**: Runtime task instantiation
4. **Data-Aware Scheduling (`Dataset`)**: Event-driven pipeline orchestration
5. **Sensors & Polling Modes**: `mode='poke'` vs `mode='reschedule'`
6. **Programmatic DAG Testing**: Inspecting DAG integrity with `DagBag`

## 1. Airflow Component Architecture

Apache Airflow is a platform to programmatically author, schedule, and monitor workflows.

```
+-------------------+     +---------------------+     +--------------------+
|  Airflow Web UI   |     |  Airflow Scheduler  |     |  Metadata Database |
|  (Flask/React)    | <-> |  (DagFileProcessor) | <-> | (PostgreSQL/MySQL) |
+-------------------+     +----------+----------+     +--------------------+
                                     |                                      
                                     v                                      
                          +---------------------+                           
                          |  Airflow Executor   |                           
                          | (Celery/Kubernetes) |                           
                          +----------+----------+                           
                                     |                                      
                                     v                                      
                          +---------------------+                           
                          |   Airflow Workers   |                           
                          |  (Task Instances)   |                           
                          +---------------------+                           
```

- **Scheduler**: Parses DAG files, evaluates schedules, and enqueues task instances into the Executor queue.
- **Webserver**: Renders the UI for monitoring DAG runs, task logs, XComs, and connection configuration.
- **Metadata Database**: Central database storing state of DAG runs, tasks, connections, variables, and XComs.
- **Executor**: Defines how tasks are executed (Sequential, Local, Celery, Kubernetes).
- **Worker**: Process/Container that pulls task instances from queue and executes code.

In [ ]:
# Code Snippet 1: Inspecting Airflow Configuration & System Context
import os
import sys
from datetime import datetime, timedelta

print(f"Python Version: {sys.version}")
print(f"Airflow Home: {os.getenv('AIRFLOW_HOME', 'Not set (using default ~/.airflow)')}")

## 2. TaskFlow API & Dynamic Task Mapping (`.expand()`)

Airflow 2.0+ introduced the TaskFlow API (`@dag`, `@task`), which simplifies data passing via automatic XCom push/pull.  
Airflow 2.3+ introduced Dynamic Task Mapping (`.expand()`), allowing DAGs to dynamically spawn task instances based on runtime lists.

In [ ]:
# Code Snippet 2: TaskFlow API Example
from airflow.decorators import dag, task

@dag(
    dag_id="demo_taskflow_notebook",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
)
def demo_pipeline():
    
    @task
    def get_partitions() -> list[str]:
        return ["2024-01-01", "2024-01-02", "2024-01-03"]
    
    @task
    def process_partition(partition_date: str) -> dict:
        print(f"Processing partition: {partition_date}")
        return {"partition": partition_date, "status": "SUCCESS"}
    
    @task
    def aggregate(results: list[dict]):
        print(f"Successfully processed {len(results)} mapped tasks.")
        
    partitions = get_partitions()
    mapped_results = process_partition.expand(partition_date=partitions)
    aggregate(results=mapped_results)

dag_obj = demo_pipeline()
print(f"Instantiated DAG ID: {dag_obj.dag_id}")
print(f"DAG Task IDs: {list(dag_obj.task_dict.keys())}")

## 3. Data-Aware Scheduling (Airflow Datasets)

Data-Aware Scheduling (introduced in Airflow 2.4+) replaces rigid cron schedules with event-driven triggers based on dataset updates.

```python
from airflow.datasets import Dataset

# 1. Define Dataset URI
RAW_ORDERS_DATASET = Dataset("s3://my-bucket/raw/orders")

# 2. Producer Task updates dataset
ingest_task = PythonOperator(
    task_id="ingest",
    python_callable=fetch_orders,
    outlets=[RAW_ORDERS_DATASET]
)

# 3. Consumer DAG triggers on Dataset update
with DAG(dag_id="consumer_dag", schedule=[RAW_ORDERS_DATASET], ...):
    transform_task = ...
```

## 4. Sensor Optimization: Poke vs. Reschedule

| Feature | `mode='poke'` | `mode='reschedule'` |
|---|---|---|
| Worker Resource Usage | Holds worker slot continuously while waiting | Releases worker slot between polling attempts |
| Memory & Task Slot Impact | High (can starve worker pool) | Low (frees worker for other tasks) |
| Best Used For | Short wait times (< 5 minutes) | Long wait times (hours/overnight file drops) |
| Airflow UI State | `running` | `up_for_reschedule` (between polls) |

In [ ]:
# Code Snippet 3: Programmatic DagBag Integrity Test
from airflow.models import DagBag

dag_folder = "../src/dags"
dagbag = DagBag(dag_folder=dag_folder, include_examples=False)

print(f"Total Discovered DAGs: {len(dagbag.dags)}")
print(f"Total Import Errors  : {len(dagbag.import_errors)}")

for dag_id, dag in dagbag.dags.items():
    print(f" - DAG: {dag_id:<35} | Tasks: {len(dag.tasks):<2} | Schedule: {dag.schedule_interval}")

if dagbag.import_errors:
    print("\nImport Error Details:")
    for filepath, error in dagbag.import_errors.items():
        print(f"File: {filepath}\nError: {error}")